In [1]:
import numpy as np
import pandas as pd
from collections import deque
import random
from tqdm import tqdm

# Configuration
SIZE = 5
FEATURES = SIZE * SIZE

In [2]:
class HexBoard:
    def __init__(self, size=SIZE):
        self.size = size
        self.board = np.zeros((size, size), dtype=int) # 0: vide, 1: Bleu, 2: Rouge

    def get_neighbors(self, r, c):
        neighbors = []
        directions = [(-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0)]
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.size and 0 <= nc < self.size:
                neighbors.append((nr, nc))
        return neighbors

    def check_win(self, player):
        # Bleu (1): Haut -> Bas | Rouge (2): Gauche -> Droite
        starts = [(0, c) for c in range(self.size) if self.board[0, c] == 1] if player == 1 \
                 else [(r, 0) for r in range(self.size) if self.board[r, 0] == 2]
        
        queue = deque(starts)
        visited = set(starts)
        while queue:
            r, c = queue.popleft()
            if (player == 1 and r == self.size - 1) or (player == 2 and c == self.size - 1):
                return True
            for move in self.get_neighbors(r, c):
                if self.board[move] == player and move not in visited:
                    visited.add(move)
                    queue.append(move)
        return False

    def encode(self):
        # Encode en 50 features (ci_b, ci_r) comme demandé [cite: 16, 19]
        encoding = []
        for val in self.board.flatten():
            encoding.extend([1, 0] if val == 1 else ([0, 1] if val == 2 else [0, 0]))
        return encoding